In [1]:
from os import getenv
import time

from dotenv import load_dotenv
from openai import OpenAI

import pandas as pd

In [2]:
load_dotenv()

OPENAI_API_KEY = getenv("OPENAI_API_KEY")

In [3]:
# 데이터 로드

df = pd.read_excel("./dataset/teacher_data.xlsx")
df.head()

,instruction,input,output
0,요즘 허리가 안 좋아서 데드는 무리하지 않게 조절해줬으면 해.,"{'성별': '남성', '몸무게': 121.05, 'squat_1RM': 215.0...",NaN
1,근지구력을 높이고 싶어. 가능하면 액세서리 운동도 같이 포함해줘.,"{'성별': '남성', '몸무게': 107.4, 'squat_1RM': 270.0,...",NaN
2,요즘 벤치가 안 늘어서 고민이야. 벤치 집중 루틴이 포함된 4주 5/3/1 프로그램...,"{'성별': '남성', '몸무게': 97.8, 'squat_1RM': 197.5, ...",NaN
3,건강 유지와 체력 향상이 목표야.,"{'성별': '남성', '몸무게': 84.8, 'squat_1RM': 260.0, ...",NaN
4,체중 감량도 병행 중이라 볼륨은 너무 많지 않았으면 해.,"{'성별': '남성', '몸무게': 86.7, 'squat_1RM': 185.0, ...",NaN


In [4]:
try:
    client = OpenAI(
        api_key=OPENAI_API_KEY
    )
except Exception as e:
    print("모델 로드 오류 : ", e)

In [5]:
prompt =f"""
### Guidelines ###
[1] 당신은 긍정적이고, 유능한 헬스 트레이너입니다.
[2] 531 by jim wendler에 기반하여, 사용자에게 운동가이드라인을 제공하고 있습니다.
[3] 당신은 각 주차 별로 사용자들에게 중량을 추천해줘야 합니다. 원판은 최소단위가 2.5kg입니다.
[4] 당신은 사용자의 1RM 입력에 대해 TM(실제 1RM의 90%)을 계산하는 과정을 답변에 포함해주세요.
[5] 사용자가 특정 부위를 더욱 강화하길 윈하면, 무게를 약간 더 증량해줘야 합니다.
[7] 사용자의 운동 경력에 맞게 답변에 사용할 운동 용어를 조절하여, 사용자가 이해하기 쉬운 수준에서 답변해주세요.
[8] 예를 들어, 초급자에게는 '1RM' -> '들 수 있는 최대 무게'처럼 쉽게 풀어 설명해주세요.
[9] 사용자들에게 동기부여해주세요. 그러면 사람들은 당신을 더욱 신뢰하고, 연봉이 상승할지도 몰라요!

### Examples ###
<instruction>
건강 유지와 체력 향상이 목표야.

<input>
'성별': '남성', '몸무게': 84.8, 'squat_1RM': 260.0, 'press_1RM': 117.2, 'bench_press_1RM': 167.5, 'deadlift_1RM': 260.0

<output>
건강 유지와 체력 향상을 목표로 하신다니 정말 멋진 목표입니다! 531 프로그램을 기반으로 주차별로 추천 중량을 안내해드릴게요. 이 프로그램은 4주 주기로 구성되어 있으며, 각 주차마다 점진적으로 중량을 증가시켜 나갑니다.

먼저, 당신의 1RM(들 수 있는 최대 무게)을 기준으로 TM(실제 1RM의 90%)을 계산해보겠습니다.

- **스쿼트 TM**: 260kg × 0.9 = 234kg
- **밀리터리 프레스 TM**: 117.2kg × 0.9 = 105.5kg
- **벤치 프레스 TM**: 167.5kg × 0.9 = 150.75kg
- **데드리프트 TM**: 260kg × 0.9 = 234kg

이제 주차별로 추천 중량을 안내해드리겠습니다.

### 1주차 (TM의 65%, 75%, 85%):
- **스쿼트**: 5회 152.5kg, 5회 175kg, 5회 이상 200kg
- **밀리터리 프레스**: 5회 67.5kg, 5회 80kg, 5회 이상 90kg
- **벤치 프레스**: 5회 97.5kg, 5회 112.5kg, 5회 이상 127.5kg
- **데드리프트**: 5회 152.5kg, 5회 175kg, 5회 이상 200kg

### 2주차 (TM의 70%, 80%, 90%):
- **스쿼트**: 3회 165kg, 3회 187.5, 3회 이상 210kg
- **밀리터리 프레스**: 3회 75kg, 3회 85kg, 3회 이상 95kg
- **벤치 프레스**: 3회 105g, 3회 120kg, 3회 이상 135kg
- **데드리프트**: 3회 165kg, 3회 187.5kg, 3회 이상 210kg

### 3주차 (TM의 75%, 85%, 95%):
- **스쿼트**: 5회 175kg, 3회 200kg, 1회 이상 222.5kg
- **밀리터리 프레스**: 5회 80kg, 3회 90kg, 1회 이상 100kg
- **벤치 프레스**: 5회 112.5kg, 3회 127.5kg, 1회 이상 142.5kg
- **데드리프트**: 5회 175kg, 3회 200kg, 1회 이상 222.5kg

### 4주차 (TM의 40%, 50%, 60%) (디로드 주):
- **스쿼트**: 5회 92.5kg, 5회 117.5kg, 5회 이상 140kg
- **밀리터리 프레스**: 5회 42.5kg, 5회 52.5kg, 5회 이상 62.5kg
- **벤치 프레스**: 5회 60kg, 5회 75kg, 5회 이상 90kg
- **데드리프트**: 5회 92.5kg, 5회 117.5kg, 5회 이상 140kg

운동을 진행하면서 자신의 몸 상태를 잘 살피고, 필요할 경우 중량을 조절하세요. 꾸준한 운동을 통해 건강과 체력을 향상시킬 수 있습니다. 당신의 목표를 응원합니다! 힘내세요! 💪
"""

In [6]:
start_time = time.time()
train_outputs =[]

for index, row in df.iterrows():
    instruction = row["instruction"]
    input = row["input"]

    user_input =f"""{instruction}
            
    [input]
    {input}
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": user_input}
            ],
            temperature=0.1
        )
        result = response.choices[0].message.content
    except Exception as e:
        print(f"index : {index} | ❌ ERROR : {e}")
        train_outputs.append(e)
    train_outputs.append(result)

    print(f"index : {index} | ✅ Good")
end_time = time.time()

elapsed_time = end_time - start_time
print(f"실행 시간: {elapsed_time:.2f}초")

index : 0 | ✅ Good
index : 1 | ✅ Good
index : 2 | ✅ Good
index : 3 | ✅ Good
index : 4 | ✅ Good
index : 5 | ✅ Good
index : 6 | ✅ Good
index : 7 | ✅ Good
index : 8 | ✅ Good
index : 9 | ✅ Good
index : 10 | ✅ Good
index : 11 | ✅ Good
index : 12 | ✅ Good
index : 13 | ✅ Good
index : 14 | ✅ Good
index : 15 | ✅ Good
index : 16 | ✅ Good
index : 17 | ✅ Good
index : 18 | ✅ Good
index : 19 | ✅ Good
index : 20 | ✅ Good
index : 21 | ✅ Good
index : 22 | ✅ Good
index : 23 | ✅ Good
index : 24 | ✅ Good
index : 25 | ✅ Good
index : 26 | ✅ Good
index : 27 | ✅ Good
index : 28 | ✅ Good
index : 29 | ✅ Good
index : 30 | ✅ Good
index : 31 | ✅ Good
index : 32 | ✅ Good
index : 33 | ✅ Good
index : 34 | ✅ Good
index : 35 | ✅ Good
index : 36 | ✅ Good
index : 37 | ✅ Good
index : 38 | ✅ Good
index : 39 | ✅ Good
index : 40 | ✅ Good
index : 41 | ✅ Good
index : 42 | ✅ Good
index : 43 | ✅ Good
index : 44 | ✅ Good
index : 45 | ✅ Good
index : 46 | ✅ Good
index : 47 | ✅ Good
index : 48 | ✅ Good
index : 49 | ✅ Good
index : 50

In [7]:
print(train_outputs[0])


허리가 안 좋으시군요. 그런 경우에는 데드리프트 중량을 조절하여 안전하게 운동하는 것이 중요합니다. 531 프로그램을 기반으로 주차별로 추천 중량을 안내해드릴게요.

먼저, 당신의 1RM(들 수 있는 최대 무게)을 기준으로 TM(실제 1RM의 90%)을 계산해보겠습니다.

- **스쿼트 TM**: 215kg × 0.9 = 193.5kg
- **밀리터리 프레스 TM**: 103.2kg × 0.9 = 92.88kg
- **벤치 프레스 TM**: 147.5kg × 0.9 = 132.75kg
- **데드리프트 TM**: 222.5kg × 0.9 = 200.25kg

이제 주차별로 추천 중량을 안내해드리겠습니다. 데드리프트는 허리를 고려하여 중량을 약간 낮춰서 설정하겠습니다.

### 1주차 (TM의 65%, 75%, 85%):
- **스쿼트**: 5회 125kg, 5회 145kg, 5회 이상 165kg
- **밀리터리 프레스**: 5회 65kg, 5회 75kg, 5회 이상 85kg
- **벤치 프레스**: 5회 95kg, 5회 110kg, 5회 이상 125kg
- **데드리프트**: 5회 130kg, 5회 150kg, 5회 이상 170kg (허리를 고려하여 중량을 조절했습니다)

### 2주차 (TM의 70%, 80%, 90%):
- **스쿼트**: 3회 135kg, 3회 155kg, 3회 이상 175kg
- **밀리터리 프레스**: 3회 70kg, 3회 80kg, 3회 이상 90kg
- **벤치 프레스**: 3회 105kg, 3회 120kg, 3회 이상 135kg
- **데드리프트**: 3회 140kg, 3회 160kg, 3회 이상 180kg (허리를 고려하여 중량을 조절했습니다)

### 3주차 (TM의 75%, 85%, 95%):
- **스쿼트**: 5회 145kg, 3회 165kg, 1회 이상 185kg
- **밀리터리 프레스**: 5회 75kg, 3회 85kg, 1회 이상 95kg
- **벤치 프레스**: 5회 110kg, 3회 125kg, 1회 이상 140

In [8]:
df["output"] = train_outputs

In [9]:
df.to_excel("./dataset/teacher_data.xlsx", index=False)